In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings("ignore")

In [2]:
fraud = pd.read_csv(
    "../data/raw/Fraud_Data.csv"
)

country = pd.read_csv(
    "../data/raw/IpAddress_to_Country.csv"
)

In [ ]:
fraud.head()

In [ ]:
country.head()

In [ ]:
fraud["signup_time"] = pd.to_datetime(
    fraud["signup_time"]
)

fraud["purchase_time"] = pd.to_datetime(
    fraud["purchase_time"]
)

In [ ]:
fraud.info()

In [ ]:
fraud["time_since_signup"] = (
    fraud["purchase_time"]
    -
    fraud["signup_time"]
).dt.total_seconds() / 3600

In [ ]:
fraud["hour_of_day"] = (
    fraud["purchase_time"]
    .dt.hour
)

In [ ]:
fraud["day_of_week"] = (
    fraud["purchase_time"]
    .dt.dayofweek
)

In [3]:
fraud[
    [
        "time_since_signup",
        "hour_of_day",
        "day_of_week"
    ]
].head()

KeyError: "None of [Index(['time_since_signup', 'hour_of_day', 'day_of_week'], dtype='str')] are in the [columns]"

In [ ]:
fraud["transaction_count"] = (
    fraud
    .groupby("user_id")
    ["user_id"]
    .transform("count")
)

In [ ]:
fraud["transactions_per_device"] = (
    fraud
    .groupby("device_id")
    ["device_id"]
    .transform("count")
)

In [ ]:
fraud[
    [
        "transaction_count",
        "transactions_per_device"
    ]
].head()

In [ ]:
fraud["ip_address"].head()

In [ ]:
country.head()

In [ ]:
fraud["ip_address"] = (
    fraud["ip_address"]
    .astype("int64")
)

In [ ]:
fraud = fraud.sort_values(
    "ip_address"
)

country = country.sort_values(
    "lower_bound_ip_address"
)

In [ ]:
fraud = pd.merge_asof(
    fraud,
    country,
    left_on="ip_address",
    right_on="lower_bound_ip_address",
    direction="backward"
)

In [ ]:
fraud = fraud[
    fraud["ip_address"]
    <=
    fraud["upper_bound_ip_address"]
]

In [ ]:
fraud["country"].value_counts().head(20)

In [ ]:
fraud[
    [
        "time_since_signup",
        "transaction_count",
        "transactions_per_device"
    ]
].describe()

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    x="class",
    y="time_since_signup",
    data=fraud
)

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    x="class",
    y="transactions_per_device",
    data=fraud
)

plt.show()

In [ ]:
fraud = pd.get_dummies(
    fraud,
    columns=[
        "source",
        "browser",
        "sex",
        "country"
    ],
    drop_first=True
)

In [ ]:
fraud.shape

In [ ]:
scaler = StandardScaler()

In [ ]:
columns_to_scale = [
    "purchase_value",
    "age",
    "time_since_signup",
    "transaction_count",
    "transactions_per_device"
]

fraud[columns_to_scale] = (
    scaler.fit_transform(
        fraud[columns_to_scale]
    )
)

In [ ]:
fraud.head()

In [ ]:
fraud.to_csv(
    "../data/processed/fraud_processed.csv",
    index=False
)

In [ ]:
print("""
TASK 1 FEATURE ENGINEERING COMPLETE

✓ Geolocation Mapping
✓ Time Features
✓ Frequency Features
✓ Device Features
✓ Encoding
✓ Scaling
✓ Processed Dataset Saved
""")